In [2]:
import pandas as pd

### load data

In [3]:
books_df = pd.read_csv("books_data.csv")
books_df.head(3)

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,Location,Age,Country
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002.0,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,"tyler, texas, usa",32.0,usa
1,276726,155061224,5,Rites of Passage,Judith Rae,2001.0,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,"seattle, washington, usa",32.0,usa
2,276727,446520802,0,The Notebook,Nicholas Sparks,1996.0,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,"h, new south wales, australia",16.0,australia


### Filter books with rating

In [4]:
books_df = books_df[books_df["Book-Rating"] > 0]

In [5]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397245 entries, 1 to 1048571
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   User-ID              397245 non-null  int64  
 1   ISBN                 397245 non-null  object 
 2   Book-Rating          397245 non-null  int64  
 3   Book-Title           351633 non-null  object 
 4   Book-Author          351633 non-null  object 
 5   Year-Of-Publication  351633 non-null  float64
 6   Publisher            351633 non-null  object 
 7   Image-URL-S          351633 non-null  object 
 8   Image-URL-M          351633 non-null  object 
 9   Image-URL-L          351633 non-null  object 
 10  Location             394990 non-null  object 
 11  Age                  394990 non-null  float64
 12  Country              375450 non-null  object 
dtypes: float64(2), int64(2), object(9)
memory usage: 42.4+ MB


In [6]:
books_df.shape

(397245, 13)

In [7]:
books_df.isnull().sum()

User-ID                    0
ISBN                       0
Book-Rating                0
Book-Title             45612
Book-Author            45612
Year-Of-Publication    45612
Publisher              45612
Image-URL-S            45612
Image-URL-M            45612
Image-URL-L            45612
Location                2255
Age                     2255
Country                21795
dtype: int64

### calculate count and average rating 

In [8]:
book_stats = books_df.groupby("Book-Title").agg(
    avg_rating=("Book-Rating", "mean"),
    rating_count=("Book-Rating", "count")
).reset_index()
book_stats.head(), book_stats.shape

(                                          Book-Title  avg_rating  rating_count
 0   A Light in the Storm: The Civil War Diary of ...    9.000000             1
 1                                       Dark Justice   10.000000             1
 2   Earth Prayers From around the World: 365 Pray...    7.142857             7
 3   Final Fantasy Anthology: Official Strategy Gu...   10.000000             2
 4   Flight of Fancy: American Heiresses (Zebra Ba...    8.000000             1,
 (128554, 3))

### sort out popular books by average rating book

In [21]:
popular_books = book_stats[book_stats["rating_count"] > 200]

popular_books = books_df.merge(popular_books, on="Book-Title")
popular_books = popular_books.drop_duplicates("Book-Title")

popular_books = popular_books.sort_values(
    by="avg_rating",
    ascending=False
)

popular_books = popular_books[
    ["Book-Title",
     "Book-Author",
     "Image-URL-M",
     "avg_rating",
     "rating_count"]
]

In [22]:
popular_books.head()

,Book-Title,Book-Author,Image-URL-M,avg_rating,rating_count
48,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,http://images.amazon.com/images/P/0439139600.0...,9.102222,225
47,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,http://images.amazon.com/images/P/0439136369.0...,9.089431,246
3,To Kill a Mockingbird,Harper Lee,http://images.amazon.com/images/P/0446310786.0...,8.962810,242
40,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,http://images.amazon.com/images/P/059035342X.0...,8.930314,287
46,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,http://images.amazon.com/images/P/0439064872.0...,8.853242,293


In [20]:
print(popular_books.columns)

Index(['User-ID', 'ISBN', 'Book-Rating', 'Book-Title', 'Book-Author',
       'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M',
       'Image-URL-L', 'Location', 'Age', 'Country', 'avg_rating',
       'rating_count'],
      dtype='object')


### filter users who rated at least 300 books

In [10]:
active_users = books_df.groupby("User-ID").count()["Book-Rating"] > 300
active_users = active_users[active_users].index

filtered_ratings = books_df[
    books_df["User-ID"].isin(active_users)
]
filtered_ratings.shape

(49413, 13)

### re-oraganise data where booktitle as a index, userID as a column, and bookrating as a values

In [11]:
pivot_table = filtered_ratings.pivot_table(
    index="Book-Title",
    columns="User-ID",
    values="Book-Rating"
).fillna(0)
#pivot_table

### find cosine similarity

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(pivot_table)
similarity_scores

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

### method for book recommendation based on partiular book

In [13]:
import numpy as np

def recommend(book_name):
    if book_name not in pivot_table.index:
        print("Book not found!")
        return

    index = np.where(pivot_table.index == book_name)[0][0]
    similar_books = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print("\nRecommended Books:\n")
    for i in similar_books:
        print(pivot_table.index[i[0]])


In [14]:
recommend("Lasher")


Recommended Books:

Cutting Edge
The Queen of the Damned (Vampire Chronicles (Paperback))
Witching Hour (Lives of the Mayfair Witches)
101 Ways to Avoid Reincarnation: Or Getting It Right the First Time
A Barefoot Doctors Manual: The American Translation of the Official Chinese Paramedical Manual


### common top book recommendation 

In [15]:
def recommend_top_books():    
    popular_books = book_stats[book_stats["rating_count"] > 350]
    recommended = popular_books.sort_values(by="avg_rating", ascending=False)

    print("\nTop Recommended Books: ", recommended.shape[0],"\n")
    #print(recommended.shape[0])
    print(recommended[['Book-Title','avg_rating','rating_count']])

In [16]:
recommend_top_books()


Top Recommended Books:  6 

                                 Book-Title  avg_rating  rating_count
110156              The Secret Life of Bees    8.511936           377
97249                     The Da Vinci Code    8.439825           457
104430            The Lovely Bones: A Novel    8.170079           635
108967  The Red Tent (Bestselling Backlist)    8.164306           353
106077           The Nanny Diaries: A Novel    7.397183           355
125394                          Wild Animus    4.442966           526


In [ ]:
import pickle

pickle.dump(popular_books, open("popular_books.pkl", "wb"))
pickle.dump(pivot_table, open("pivot_table.pkl", "wb"))

In [24]:
pickle.dump(books_df, open("books.pkl", "wb"))